In [0]:
%run ./utils/logger

In [0]:
run_id = get_run_id()
print(run_id)


In [0]:
dbutils.widgets.text('catalog','commerce_stage_dev')
dbutils.widgets.text('schema','silver')
dbutils.widgets.text("env", "dev")

In [0]:
# Set default catalog and schema
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

In [0]:
env = dbutils.widgets.get("env")

In [0]:
orders_df = spark.table(f'commerce_raw_{env}.bronze.orders')

In [0]:
display(orders_df)

In [0]:
from pyspark.sql.functions import col
orders_filter_df = orders_df.filter(col("order_id").isNotNull())

In [0]:
orders_clean_df = orders_filter_df.distinct()

In [0]:
orders_clean_df.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{schema}.orders_temp1")

In [0]:
%sql
select * from orders_temp1

In [0]:
spark.sql(f"""
CREATE TABLE if not EXISTS {catalog}.{schema}.orders_stage_dedup as
select *  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY order_id
               ORDER BY ingestion_time DESC
           ) AS rn
    FROM orders_temp1
)
WHERE rn = 1
""");


In [0]:
df = spark.sql(f'describe table extended commerce_stage_{env}.{schema}.order_stage')
display(df)

In [0]:
spark.sql(f"""
MERGE INTO commerce_stage_{env}.{schema}.order_stage os
USING commerce_stage_{env}.{schema}.orders_stage_dedup od

ON os.order_id = od.order_id

WHEN MATCHED THEN
UPDATE SET
    os.customer_id = od.customer_id,
    os.order_date = od.order_date,
    os.order_status = od.order_status,
    os.payment_method = od.payment_method,
    os.updated_ts = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
    order_id,
    customer_id,
    order_date,
    order_status,
    payment_method,
    total_amount,
    created_ts,
    updated_ts
)
VALUES (
    od.order_id,
    od.customer_id,
    od.order_date,
    od.order_status,
    od.payment_method,
    NULL,
    current_timestamp(),
    current_timestamp()
)
""")

In [0]:
spark.sql(f"""
          drop table if exists commerce_stage_{env}.{schema}.orders_temp1
          """)

In [0]:
spark.sql(f"""
          drop table if exists commerce_stage_{env}.{schema}.orders_stage_dedup
          """)